# Raw Data Download

Download raw experimental data files from the aimdl/datafiles API with configurable data types.

This notebook demonstrates how to:
- Query files by data type using the aimdl/datafiles endpoint
- Download multiple files in parallel
- Save to disk with metadata

In [1]:
import sys
from pathlib import Path

import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import get_girder_client, download_item_to_disk, paginate_datafiles, get_output_dir

## Configuration

In [ ]:
from dotenv import load_dotenv

load_dotenv('..')

# Configure data type to download
# OPTIONS: "pdv_trace", "pdv_alpss_output", or "pdv_experiment_log"
DATA_TYPE = "pdv_trace"

# For pdv_alpss_output, 
# 1) Leave as empty string to download all, OR 
# 2) set FILENAME_FILTER. options are "velocity", "velocity--smooth", "noisefrac", 
# "voltage", "veluncert", "hel", "iq", "inputs", "plots"
FILENAME_FILTER = ""

# Output directory for downloaded files
OUTPUT_DIR = f"./{DATA_TYPE}"
if DATA_TYPE == "pdv_alpss_output" and FILENAME_FILTER:
    OUTPUT_DIR = f"{OUTPUT_DIR}/{FILENAME_FILTER}"

## Download Files

In [3]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    
    # Set up filter if downloading pdv_alpss_output with a filename filter
    item_filter = None
    if DATA_TYPE == "pdv_alpss_output" and FILENAME_FILTER:
        def item_filter(item):
            return FILENAME_FILTER in item.get("name", "")
    
    # Download files
    results = paginate_datafiles(
        gc,
        DATA_TYPE,
        worker_fn=lambda item: download_item_to_disk(
            gc,
            item,
            get_output_dir(item, OUTPUT_DIR)
        ),
        item_filter=item_filter
    )
    
    # Print summary
    for result in results:
        print(f"Downloaded: {result['name']} ({result['size'] / 1024:.2f} KB)")
    
    print(f"\nTotal files downloaded: {len(results)}")
    print(f"Total size: {sum(r['size'] for r in results) / (1024**2):.2f} MB")

KeyboardInterrupt: 

## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"Downloaded {len(df)} files of type '{DATA_TYPE}'")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\\nSample files:")
print(df[["name", "size"]].head(10))

Downloaded 220 files of type 'pdv_experiment_log'
Output directory: ./pdv_experiment_log
\nSample files:
                                                name   size
0     experiment_data_LMI AlMg_20250807_Al_F416.xlsx   8672
1  Cu_Ridge_height_0_20260528_JHBMAB00003-S2R2C3.csv   6370
2  Cu_ridge_height_test_20260528_JHBMAB00003-S2R5...   6268
3  experiment_data_LMI-Al-Mg-Shots_20250811_Al_F4...  12963
4  experiment_data_LMI-Al-Mg-Shots_20250811_Al_F4...  12963
5  experiment_data_LMI-Al-Mg-Shots_20250811_Al_F4...  12932
6  experiment_data_LMI-Al-Mg-Shots_20250807_Al_F4...   8702
7  experiment_data_LMI-Al-Mg-Shots_20250811_Al_FT...   5854
8  experiment_data_LMI-Al-Mg-Shots_20250807_Al_F4...   8727
9  experiment_data_LMI-Cu-Ti-Shots_20250811_Al_F4...  12963
